# Patient diagnosis and procedure code inspection

This notebook manually inspects the 25 synthetic patients and prepares their clinical codes for a future billing workflow.

It separates:

- **Current/recorded diagnosis candidates** — SNOMED CT codes from encounter `Condition` resources and encounter reasons.
- **Current/recorded procedure candidates** — SNOMED CT codes from `Procedure` resources and encounter type.
- **Future coding candidates** — recommendations extracted from the note/after-visit summary. These are text, not billing codes, and require human review before mapping.

> **Important:** the dataset does not contain ICD-10-CM, CPT, HCPCS, claims, charges, or validated billing codes. SNOMED-to-ICD/CPT mapping is not one-to-one and should not be inferred without an approved terminology service and coding review.

In [6]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

DATASET_NAME = "synthetic-ambient-fhir-25.jsonl"
DATASET_CANDIDATES = [
    Path("data/synthetic-ambient-fhir-25") / DATASET_NAME,
    Path("../data/synthetic-ambient-fhir-25") / DATASET_NAME,
]

DATASET_PATH = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    checked = "\n".join(f"- {path.resolve()}" for path in DATASET_CANDIDATES)
    raise FileNotFoundError(f"Dataset not found. Checked:\n{checked}")

with DATASET_PATH.open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

assert len(records) == 25, f"Expected 25 records, found {len(records)}"
print(f"Loaded {len(records)} patient encounters from {DATASET_PATH.resolve()}")

Loaded 25 patient encounters from /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data/synthetic-ambient-fhir-25/synthetic-ambient-fhir-25.jsonl


In [7]:
def patient_name(record):
    names = record["patient_context"]["patient"].get("name", [])
    if not names:
        return "Unknown"
    name = names[0]
    return " ".join([*name.get("given", []), name.get("family", "")]).strip()


def first_coding(codeable_concept):
    """Return the first coding plus the human-readable concept text."""
    codeable_concept = codeable_concept or {}
    coding = (codeable_concept.get("coding") or [{}])[0]
    return {
        "system": coding.get("system"),
        "code": coding.get("code"),
        "display": coding.get("display") or codeable_concept.get("text"),
        "text": codeable_concept.get("text"),
    }


def extract_code_rows(record, patient_number):
    """Extract diagnosis/procedure source codes that may support later billing review."""
    encounter = record["encounter_fhir"]["encounter"]
    related = record["encounter_fhir"].get("related_resources", {})
    base = {
        "patient_number": patient_number,
        "patient": patient_name(record),
        "encounter_date": record["metadata"]["date"][:10],
    }
    rows = []

    def add(category, source, concept, status=None, date=None):
        coding = first_coding(concept)
        rows.append({
            **base,
            "category": category,
            "source": source,
            **coding,
            "status": status,
            "resource_date": date,
        })

    for concept in encounter.get("type", []):
        add("procedure", "Encounter.type", concept, encounter.get("status"), encounter.get("period", {}).get("start"))
    for concept in encounter.get("reasonCode", []):
        add("diagnosis", "Encounter.reasonCode", concept, encounter.get("status"), encounter.get("period", {}).get("start"))

    for condition in related.get("Condition", []):
        add(
            "diagnosis",
            "Condition.code",
            condition.get("code"),
            condition.get("clinicalStatus", {}).get("coding", [{}])[0].get("code"),
            condition.get("onsetDateTime") or condition.get("recordedDate"),
        )

    for procedure in related.get("Procedure", []):
        add(
            "procedure",
            "Procedure.code",
            procedure.get("code"),
            procedure.get("status"),
            procedure.get("performedDateTime") or procedure.get("performedPeriod", {}).get("start"),
        )

    return rows


all_code_rows = [
    row
    for patient_number, record in enumerate(records)
    for row in extract_code_rows(record, patient_number)
]
codes_df = pd.DataFrame(all_code_rows).drop_duplicates(
    subset=["patient_number", "category", "source", "system", "code", "display"]
)

print(f"Extracted {len(codes_df)} unique patient/code/source rows")
display(codes_df["system"].value_counts(dropna=False).rename("rows").to_frame())

Extracted 317 unique patient/code/source rows


,rows
system,
http://snomed.info/sct,317


## Cohort index

Use this table to choose a `PATIENT_NUMBER` for detailed manual inspection. Counts represent source-code candidates, not final billable codes.

In [8]:
patient_rows = []
for patient_number, record in enumerate(records):
    patient = record["patient_context"]["patient"]
    encounter = record["encounter_fhir"]["encounter"]
    patient_codes = codes_df[codes_df["patient_number"] == patient_number]
    patient_rows.append({
        "patient_number": patient_number,
        "patient": patient_name(record),
        "gender": patient.get("gender"),
        "birth_date": patient.get("birthDate"),
        "encounter_date": record["metadata"]["date"][:10],
        "visit_title": record["metadata"]["visit_title"],
        "facility": encounter.get("serviceProvider", {}).get("display"),
        "diagnosis_candidates": (patient_codes["category"] == "diagnosis").sum(),
        "procedure_candidates": (patient_codes["category"] == "procedure").sum(),
    })

patients_df = pd.DataFrame(patient_rows)
display(patients_df)

,patient_number,patient,gender,birth_date,encounter_date,visit_title,facility,diagnosis_candidates,procedure_candidates
0,0,Ali Jerry Kuhic,male,1991-05-24,2022-08-05,Annual physical — preventive screening and migraine check-in,WHITLEY WELLNESS LLC,3,7
1,1,Ariane Jan Runolfsson,female,1940-05-16,2021-01-03,Inpatient admission — COVID-19 isolation with pneumonia and hypoxemia,"FIRST PSYCHIATRIC PLANNERS, INC",4,4
2,2,Clarence Reinger,female,1993-05-09,2025-06-01,Prenatal intake visit — initial obstetric evaluation,CAMBRIDGE PUBLIC HEALTH COMMISSION,2,21
3,3,Corinne Charlesetta Crooks,female,1970-12-13,2017-01-01,Annual wellness examination — preventive screening and health maintenance,"LEAP INTO WELLNESS,LLC",2,6
4,4,Delana Jamie Gutkowski,female,2003-03-21,2025-05-23,Young adult preventive exam — prediabetes and allergy follow-up,MEASURED WELLNESS LLC,4,9
5,5,Dick Larson,male,1987-05-23,2024-05-18,Annual check-up — post-sepsis recovery and prediabetes,CHILDREN'S HOSPITAL CORPORATION,3,6
6,6,Elias Wisozk,male,1972-05-25,2026-06-18,General adult exam — new hypertension and metabolic syndrome,GREATER LAWRENCE FAMILY HEALTH CENTER INC,5,8
7,7,Emory Britt Kovacek,male,1999-01-19,2021-04-06,General exam — chronic low back pain and positive depression screen,SUNRISE HEALTHCARE LLC,3,9
8,8,Eva Julia Casas,female,1954-06-22,2016-08-30,"Annual general exam — prediabetes, hyperlipidemia, and knee osteoarthritis","PRIMARY CARE MEDICINE AND PEDIATRICS, LLC",2,9
9,9,Hiedi Moira Thiel,female,1993-07-26,2024-07-22,"Prenatal intake visit — first trimester, newly identified anemia",ENCOMPASS HEALTH REHAB HOSPITAL OF WESTERN MASS,3,21


## Inspect one patient

Change `PATIENT_NUMBER` (0–24) and rerun this cell. The structured tables are current encounter evidence. The future-plan section intentionally remains text because the dataset supplies no future ICD-10/CPT codes.

In [9]:
PATIENT_NUMBER = 0  # choose 0 through 24


def assessment_and_plan(note):
    match = re.search(
        r"(?is)(?:\*\*)?assessment\s*(?:&|and)\s*plan(?:\*\*)?\s*:?\s*(.*)$",
        note,
    )
    return match.group(1).strip() if match else "Assessment & Plan section not detected; inspect the full note below."


def future_action_lines(text):
    """Surface likely future actions for review; this is not a clinical coding model."""
    action_pattern = re.compile(
        r"\b(start|begin|order|refer|schedule|follow[- ]?up|return|repeat|continue|stop|discontinue|"
        r"increase|decrease|monitor|recommend|plan|prescrib|consider|arrange|check)\w*\b",
        re.IGNORECASE,
    )
    lines = [re.sub(r"^[•*\-\s]+", "", line).strip() for line in text.splitlines()]
    return [line for line in lines if line and action_pattern.search(line)]


record = records[PATIENT_NUMBER]
patient = record["patient_context"]["patient"]
encounter = record["encounter_fhir"]["encounter"]
selected_codes = codes_df[codes_df["patient_number"] == PATIENT_NUMBER].copy()

summary = {
    "Patient number": PATIENT_NUMBER,
    "Patient": patient_name(record),
    "Gender": patient.get("gender"),
    "Birth date": patient.get("birthDate"),
    "Encounter date": record["metadata"]["date"][:10],
    "Visit": record["metadata"]["visit_title"],
    "Facility": encounter.get("serviceProvider", {}).get("display"),
    "Encounter class": encounter.get("class", {}).get("code"),
}
display(pd.DataFrame(summary.items(), columns=["field", "value"]))

display(Markdown("### Current encounter diagnosis candidates"))
display(
    selected_codes[selected_codes["category"] == "diagnosis"]
    [["source", "system", "code", "display", "status", "resource_date"]]
    .reset_index(drop=True)
)

display(Markdown("### Current encounter procedure candidates"))
display(
    selected_codes[selected_codes["category"] == "procedure"]
    [["source", "system", "code", "display", "status", "resource_date"]]
    .reset_index(drop=True)
)

display(Markdown("### Historical chart condition labels (labels only; source codes are not included)"))
condition_labels = record["patient_context"]["longitudinal_summary"].get("condition_labels", [])
display(pd.DataFrame({"condition_label": condition_labels}))

plan = assessment_and_plan(record["note"])
actions = future_action_lines(plan + "\n" + record["after_visit_summary"])
display(Markdown("### Possible future coding candidates — manual review required"))
display(pd.DataFrame({"future_plan_text": actions}).drop_duplicates().reset_index(drop=True))

display(Markdown("### Assessment & Plan"))
display(Markdown(plan))
display(Markdown("### After-visit summary"))
display(Markdown(record["after_visit_summary"]))

,field,value
0,Patient number,0
1,Patient,Ali Jerry Kuhic
2,Gender,male
3,Birth date,1991-05-24
4,Encounter date,2022-08-05
5,Visit,Annual physical — preventive screening and migraine check-in
6,Facility,WHITLEY WELLNESS LLC
7,Encounter class,AMB


### Current encounter diagnosis candidates

,source,system,code,display,status,resource_date
0,Condition.code,http://snomed.info/sct,160903007,Full-time employment (finding),resolved,2022-08-05T06:06:12-07:00
1,Condition.code,http://snomed.info/sct,422650009,Social isolation (finding),active,2022-08-05T06:06:12-07:00
2,Condition.code,http://snomed.info/sct,66383009,Gingivitis (disorder),resolved,2022-08-05T07:10:04-07:00


### Current encounter procedure candidates

,source,system,code,display,status,resource_date
0,Encounter.type,http://snomed.info/sct,162673000,General examination of patient (procedure),finished,2022-08-05T05:19:56-07:00
1,Procedure.code,http://snomed.info/sct,430193006,Medication reconciliation (procedure),completed,2022-08-05T05:19:56-07:00
2,Procedure.code,http://snomed.info/sct,710824005,Assessment of health and social care needs (procedure),completed,2022-08-05T05:19:56-07:00
3,Procedure.code,http://snomed.info/sct,866148006,Screening for domestic abuse (procedure),completed,2022-08-05T06:06:12-07:00
4,Procedure.code,http://snomed.info/sct,428211000124100,Assessment of substance use (procedure),completed,2022-08-05T06:35:36-07:00
5,Procedure.code,http://snomed.info/sct,713106006,Screening for drug abuse (procedure),completed,2022-08-05T06:49:31-07:00
6,Procedure.code,http://snomed.info/sct,103697008,Patient referral for dental care (procedure),completed,2022-08-05T07:10:04-07:00


...

,future_plan_text
0,Patient referral for dental care placed (sliding-scale clinic).
1,Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.
2,Transport problem and lack of access to transportation continue to interfere with appointments.
3,Lifestyle counseling on diet and regular aerobic activity; repeat lipid panel to follow the trend.
4,"Return visit in one year, or sooner as needed."


### Assessment & Plan

**

### Gingivitis
Gingival inflammation with easy bleeding on exam; years since last dental care, with cost and transportation as barriers.
- Patient referral for dental care placed (sliding-scale clinic).
- Daily flossing and twice-daily brushing reviewed.

### Chronic intractable migraine without aura
Stable long-standing pattern, about two attacks weekly, no red-flag features; pain 2/10 today.
- Medication reconciliation completed and chart list updated.
- Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.

### Social isolation and stress
Newly documented social isolation; significant financial stress and short sleep, without features of a depressive disorder today.
- Assessment of health and social care needs completed (PRAPARE).
- Care coordinator outreach this week; encouraged trial of community men's group.

### Transportation insecurity
Transport problem and lack of access to transportation continue to interfere with appointments.
- Enrollment in medical ride-service program via care coordination, including dental visits.

### Preventive health and screening
Lipid panel shows LDL 137.19 mg/dL and HDL 33.25 mg/dL; domestic-abuse screening (HARK 0), substance use assessment, and drug-abuse screening (DAST-10 0) were all reassuring; never-smoker.
- Lifestyle counseling on diet and regular aerobic activity; repeat lipid panel to follow the trend.
- Influenza (split virus, trivalent, PF) and Td booster administered today.
- Helmet and protective-gear counseling for recreational ramp riding.
- Return visit in one year, or sooner as needed.

### After-visit summary

Visit summary

What we discussed
• Gingivitis
• Chronic intractable migraine without aura
• Social isolation and stress
• Transportation insecurity
• Preventive health and screening

Next steps
• Patient referral for dental care placed (sliding-scale clinic).
• Daily flossing and twice-daily brushing reviewed.
• Medication reconciliation completed and chart list updated.
• Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.
• Assessment of health and social care needs completed (PRAPARE).
• Care coordinator outreach this week; encouraged trial of community men's group.
• Enrollment in medical ride-service program via care coordination, including dental visits.
• Lifestyle counseling on diet and regular aerobic activity; repeat lipid panel to follow the trend.

## Cohort-wide review and optional export

These tables preserve the source terminology and evidence. Do not relabel them as ICD-10/CPT without a validated mapping workflow.

In [10]:
future_rows = []
for patient_number, record in enumerate(records):
    plan = assessment_and_plan(record["note"])
    actions = future_action_lines(plan + "\n" + record["after_visit_summary"])
    for action in dict.fromkeys(actions):
        future_rows.append({
            "patient_number": patient_number,
            "patient": patient_name(record),
            "encounter_date": record["metadata"]["date"][:10],
            "future_plan_text": action,
            "coding_status": "uncoded_text_requires_review",
        })

future_candidates_df = pd.DataFrame(future_rows)

print("Structured diagnosis/procedure source codes")
display(
    codes_df.sort_values(["patient_number", "category", "display"])
    .reset_index(drop=True)
)

print("Future plan text requiring manual coding review")
display(future_candidates_df)

# Reproducible processed lookup: one row per patient/source code.
patient_ids = {i: record["metadata"]["patient_id"] for i, record in enumerate(records)}
encounter_ids = {i: record["metadata"]["encounter_id"] for i, record in enumerate(records)}
patient_codes_export_df = codes_df.copy()
patient_codes_export_df.insert(
    1, "patient_id", patient_codes_export_df["patient_number"].map(patient_ids)
)
patient_codes_export_df.insert(
    2, "encounter_id", patient_codes_export_df["patient_number"].map(encounter_ids)
)
patient_codes_export_df = patient_codes_export_df.sort_values(
    ["patient_number", "category", "display"]
).reset_index(drop=True)

processing_dir = Path("data_processing") if Path("data_processing").is_dir() else Path(".")
output_dir = processing_dir / "processed_csv"
output_dir.mkdir(parents=True, exist_ok=True)
patient_codes_path = output_dir / "patient_to_codes.csv"
patient_codes_export_df.to_csv(patient_codes_path, index=False)
print(f"Wrote {len(patient_codes_export_df)} rows to {patient_codes_path.resolve()}")

Structured diagnosis/procedure source codes


,patient_number,patient,encounter_date,category,source,system,code,display,text,status,resource_date
0,0,Ali Jerry Kuhic,2022-08-05,diagnosis,Condition.code,http://snomed.info/sct,160903007,Full-time employment (finding),Full-time employment (finding),resolved,2022-08-05T06:06:12-07:00
1,0,Ali Jerry Kuhic,2022-08-05,diagnosis,Condition.code,http://snomed.info/sct,66383009,Gingivitis (disorder),Gingivitis (disorder),resolved,2022-08-05T07:10:04-07:00
2,0,Ali Jerry Kuhic,2022-08-05,diagnosis,Condition.code,http://snomed.info/sct,422650009,Social isolation (finding),Social isolation (finding),active,2022-08-05T06:06:12-07:00
3,0,Ali Jerry Kuhic,2022-08-05,procedure,Procedure.code,http://snomed.info/sct,710824005,Assessment of health and social care needs (procedure),Assessment of health and social care needs (procedure),completed,2022-08-05T05:19:56-07:00
4,0,Ali Jerry Kuhic,2022-08-05,procedure,Procedure.code,http://snomed.info/sct,428211000124100,Assessment of substance use (procedure),Assessment of substance use (procedure),completed,2022-08-05T06:35:36-07:00
...,...,...,...,...,...,...,...,...,...,...,...
312,24,Van Loren O'Reilly,2019-07-25,procedure,Procedure.code,http://snomed.info/sct,171207006,Depression screening (procedure),Depression screening (procedure),completed,2019-07-25T09:11:38-07:00
313,24,Van Loren O'Reilly,2019-07-25,procedure,Procedure.code,http://snomed.info/sct,715252007,Depression screening using Patient Health Questionnaire Nine Item score (procedure),Depression screening using Patient Health Questionnaire Nine Item score (procedure),completed,2019-07-25T09:53:05-07:00
314,24,Van Loren O'Reilly,2019-07-25,procedure,Encounter.type,http://snomed.info/sct,162673000,General examination of patient (procedure),General examination of patient (procedure),finished,2019-07-25T07:59:19-07:00
315,24,Van Loren O'Reilly,2019-07-25,procedure,Procedure.code,http://snomed.info/sct,430193006,Medication reconciliation (procedure),Medication reconciliation (procedure),completed,2019-07-25T07:59:19-07:00


Future plan text requiring manual coding review


,patient_number,patient,encounter_date,future_plan_text,coding_status
0,0,Ali Jerry Kuhic,2022-08-05,Patient referral for dental care placed (sliding-scale clinic).,uncoded_text_requires_review
1,0,Ali Jerry Kuhic,2022-08-05,Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.,uncoded_text_requires_review
2,0,Ali Jerry Kuhic,2022-08-05,Transport problem and lack of access to transportation continue to interfere with appointments.,uncoded_text_requires_review
3,0,Ali Jerry Kuhic,2022-08-05,Lifestyle counseling on diet and regular aerobic activity; repeat lipid panel to follow the trend.,uncoded_text_requires_review
4,0,Ali Jerry Kuhic,2022-08-05,"Return visit in one year, or sooner as needed.",uncoded_text_requires_review
...,...,...,...,...,...
196,24,Van Loren O'Reilly,2019-07-25,Behavioral plan discussed: brisk 20-30 minute walks most evenings; scheduled daily worry time for budgeting.,uncoded_text_requires_review
197,24,Van Loren O'Reilly,2019-07-25,Provided information for sliding-scale counseling; patient will consider — not saying no.,uncoded_text_requires_review
198,24,Van Loren O'Reilly,2019-07-25,"No medication initiated today; return in 2-3 months to reassess, sooner if worry disrupts sleep nightly or chest tig...",uncoded_text_requires_review
199,24,Van Loren O'Reilly,2019-07-25,"Continue naproxen sodium 220 mg as needed, always with food; monitor for GI symptoms.",uncoded_text_requires_review


Wrote 317 rows to /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data_processing/processed_csv/patient_to_codes.csv
